<a href="https://colab.research.google.com/github/carpotru08-cmd/Introducci-n_Ciencia_Datos_UdeM/blob/main/Trabajo%20Final/ML_Rentabilidad_Segunda_Entrega.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelo unificado de rentabilidad — Ventas 2022–2025
## Workbook acumulativo: EDA → Preprocesamiento → Regresión → Interpretación de negocio

### Pregunta de negocio

> **¿Qué características de las ventas, productos, clientes, canales y periodos están asociadas con una mayor rentabilidad y cuáles tienen mayor capacidad para explicar o predecir el margen de ganancia?**

La rentabilidad se representa mediante:

`Profit_Margin`

---

## Decisión principal: regresión

`Profit_Margin` es una variable **numérica continua**.

Por ejemplo, una transacción puede tener:

- 4.2 % de margen;
- 13.8 %;
- 19.7 %;
- 27.4 %.

Por eso utilizaremos:

### Aprendizaje supervisado → Regresión

**¿Por qué no convertirla directamente en clasificación?**

Porque una clasificación como:

`Profit_Margin < 15 % → baja rentabilidad`

haría que un margen de 2 % y uno de 14.9 % se conviertan en la misma clase, perdiendo información.

La regresión permite responder mejor la pregunta original: **cuánto cambia la rentabilidad y qué variables ayudan a explicarla**.

Al final del notebook convertiremos la predicción continua en una **alerta operativa de baja rentabilidad**, sin cambiar el objetivo principal del modelo.

# 0. Estrategia metodológica

Este workbook conserva de forma acumulativa las cuatro etapas de preprocesamiento vistas en clase:

1. **Imputación y Missing Indicator**
2. **Codificación de variables categóricas**
3. **Escalado y normalización**
4. **Texto y vectorización**

Después incorpora una etapa de modelado con varias buenas prácticas:

- un modelo `Dummy` como baseline;
- comparación de varias familias de regresión;
- validación cruzada solamente en entrenamiento;
- conjunto de test reservado hasta el final;
- principio de parsimonia: no elegir un modelo más complejo por una mejora mínima;
- verificación adicional de leakage si el desempeño es demasiado alto;
- importancia por permutación;
- análisis de errores por segmentos;
- simulación de descuentos por producto;
- traducción del margen predicho a una alerta de negocio.

> En todo momento distinguiremos **asociación predictiva** de causalidad.

# 1. Librerías

In [1]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone

from sklearn.model_selection import (
    train_test_split,
    KFold,
    cross_validate
)

from sklearn.impute import (
    SimpleImputer,
    KNNImputer,
    MissingIndicator
)

from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
    Normalizer
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.feature_extraction.text import (
    CountVectorizer,
    TfidfVectorizer
)

from sklearn.dummy import DummyRegressor

from sklearn.linear_model import (
    LinearRegression,
    Ridge
)

from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

RANDOM_STATE = 42

# 2. Carga de datos

La carga se mantiene intencionalmente simple.

Necesitamos:

- el dataset limpio producido por el EDA;
- el dataset original para recuperar los valores faltantes que existían antes de la limpieza.

In [2]:
df = pd.read_csv(
    "/content/Sales_transactions_2022_2025_limpio.csv"
)

df_original = pd.read_csv(
    "/content/Sales_transactions_2022_2025.csv"
)

print("Dataset limpio   :", df.shape)
print("Dataset original :", df_original.shape)

display(df.head())

Dataset limpio   : (18000, 47)
Dataset original : (18045, 36)


,Transaction_ID,Order_ID,Customer_ID,Customer_Name,Customer_Age,Customer_Gender,Customer_Segment,Order_Date,Order_Time,Sales_Channel,Store_ID,Store_Name,Country,Region,City,Product_ID,Product_Name,Product_Category,Product_Subcategory,Quantity,Unit_Price,Discount_Percentage,Sales_Amount,Cost_Amount,Profit,Payment_Method,Order_Status,Shipping_Method,Delivery_Days,Return_Flag,Return_Reason,Sales_Representative,Promotion_Code,Customer_Rating,Inventory_Level,Order_Year,Profit_Margin,Gross_Sales,Discount_Amount,Has_Promotion,Order_Month,Order_Quarter,Year_Quarter,Day_of_Week,Age_Group,Is_Returned,Period_Group
0,T02016694,O0116694,C01482,Sophia Smith,28,Male,Consumer,2022-10-06,11:30:55,Online,LA-01,Los Angeles Downtown,United States,West,Los Angeles,P026,Webcam HD,Electronics,Cameras,2,85.29,5.0,162.05,115.61,46.44,Credit Card,Completed,Standard,4,No,Not Applicable,James Anderson,SAVE10,4.0,81,2022,28.657822,170.58,8.53,1,10,4,2022-Q4,Thursday,26-35,0,Q4
1,T02016374,O0108187,C01591,Hannah King,46,Female,Consumer,2024-06-03,17:06:29,Online,SYD-01,Sydney CBD,Australia,New South Wales,Sydney,P005,Office Keyboard,Electronics,Accessories,6,58.85,15.0,300.13,237.00,63.13,Debit Card,Completed,Standard,6,No,Not Applicable,Ella Lewis,No Promotion,5.0,123,2024,21.034219,353.10,52.97,0,6,2,2024-Q2,Monday,46-55,0,Q1-Q3
2,T02004099,O0104099,C04140,Emma Wilson,42,Male,Consumer,2024-12-19,08:22:11,Online,PAR-01,Paris Central,France,Île-de-France,Paris,P017,Copy Paper Box,Office Supplies,Paper,2,43.92,15.0,74.66,73.17,1.49,Credit Card,Shipped,Standard,4,No,Not Applicable,Liam Davis,No Promotion,4.0,93,2024,1.995714,87.84,13.18,0,12,4,2024-Q4,Thursday,36-45,0,Q4
3,T02010280,O0110280,C03663,Daniel Clark,18,Male,Corporate,2025-10-05,14:32:43,B2B Portal,NYC-01,New York Central,United States,Northeast,New York,P008,SmartDock USB-C,Electronics,Accessories,11,115.34,15.0,1078.46,897.91,180.55,PayPal,Completed,Next Day,2,No,Not Applicable,Grace Harris,HOLIDAY20,NaN,59,2025,16.741465,1268.74,190.28,1,10,4,2025-Q4,Sunday,18-25,0,Q4
4,T02012685,O0112685,C02749,Sophia Smith,37,Male,Returning Customer,2022-09-13,14:55:16,B2B Portal,CHI-01,Chicago Loop,United States,Midwest,Chicago,P013,Desk Lamp,Furniture,Lighting,4,69.71,15.0,237.03,164.95,72.08,Apple Pay,Completed,Express,2,No,Not Applicable,Sophia Brown,No Promotion,2.0,56,2022,30.409653,278.84,41.81,0,9,3,2022-Q3,Tuesday,36-45,0,Q1-Q3


# 3. Puente entre el EDA y Machine Learning

Antes de construir el modelo verificamos dos ideas centrales del EDA:

1. la rentabilidad debe medirse con un **margen agregado correcto**;
2. Q4 mostró más volumen y descuentos, pero menor eficiencia.

El margen agregado se calcula como:

`100 × suma(Profit) / suma(Sales_Amount)`

No utilizamos simplemente la media de los márgenes individuales porque transacciones de tamaños muy distintos no deberían pesar igual.

In [3]:
def margen_agregado(datos):
    ventas = datos["Sales_Amount"].sum()

    if ventas == 0:
        return np.nan

    return (
        100
        * datos["Profit"].sum()
        / ventas
    )


print(
    "Margen agregado global:",
    round(margen_agregado(df), 2),
    "%"
)

Margen agregado global: 19.98 %


In [4]:
if "Order_Quarter" in df.columns:

    puente_q = (
        df
        .assign(
            Grupo_Periodo=np.where(
                df["Order_Quarter"].astype(str) == "4",
                "Q4",
                "Q1-Q3"
            )
        )
        .groupby("Grupo_Periodo")
        .agg(
            Transacciones=("Sales_Amount", "size"),
            Ventas=("Sales_Amount", "sum"),
            Utilidad=("Profit", "sum"),
            Descuento_Promedio=("Discount_Percentage", "mean")
        )
    )

    margen_q = (
        df
        .assign(
            Grupo_Periodo=np.where(
                df["Order_Quarter"].astype(str) == "4",
                "Q4",
                "Q1-Q3"
            )
        )
        .groupby("Grupo_Periodo")
        .apply(margen_agregado)
        .rename("Margen_Agregado")
    )

    puente_q = puente_q.join(margen_q)

    display(
        puente_q.round(2)
    )

,Transacciones,Ventas,Utilidad,Descuento_Promedio,Margen_Agregado
Grupo_Periodo,,,,,
Q1-Q3,12134,5536553.83,1211467.31,4.00,21.88
Q4,5866,2697536.77,433972.09,10.91,16.09


Este bloque no pretende repetir todo el EDA.

Su función es conectar el análisis exploratorio con la siguiente pregunta:

> Si Q4 presenta menor margen, ¿el trimestre realmente aporta información propia o la diferencia puede explicarse mejor por variables como producto y descuento?

Eso se estudiará con el modelo multivariado.

# 4. Selección de variables

Una buena predicción no sirve si utiliza información que no estaría disponible al momento de analizar la venta.

Por eso dividimos las variables en tres grupos.

## 4.1 Variables prohibidas por leakage

No usamos:

- `Profit`
- `Sales_Amount`
- `Cost_Amount`
- `Gross_Sales`
- `Discount_Amount`

**¿Por qué?**

`Profit_Margin` está estructuralmente ligado a esas variables. Incluirlas permitiría al modelo reconstruir la respuesta casi directamente.

## 4.2 Variables posteriores a la venta

No usamos:

- `Return_Flag`
- `Return_Reason`
- `Is_Returned`
- `Delivery_Days`
- `Customer_Rating`
- `Order_Status`

**¿Por qué?**

Si queremos utilizar el modelo antes o al momento de aprobar una venta, esta información todavía no existe.

## 4.3 Identificadores

Tampoco usamos IDs ni nombres de clientes.

No representan una característica de negocio generalizable y pueden favorecer memorización.

## 4.4 Variables seleccionadas

Para evitar redundancias mantenemos una representación compacta:

### Numéricas

- `Discount_Percentage`
- `Quantity`
- `Unit_Price`
- `Inventory_Level`

### Ordinal

- `Age_Group`

### Nominales

- producto y categoría;
- canal;
- segmento y género;
- país;
- envío y pago;
- promoción;
- trimestre, año y día.

### Decisiones de simplificación

- usamos `Age_Group` en lugar de `Customer_Age`;
- usamos `Country` y no simultáneamente `Region` y `City`;
- usamos `Order_Quarter` y no `Order_Month`;
- incluimos `Product_Name` porque el producto puede tener una estructura de costos y margen propia;
- `Product_Name` se tratará como **categoría**, no como texto libre.

In [5]:
target = "Profit_Margin"

numeric_features = [
    "Discount_Percentage",
    "Quantity",
    "Unit_Price",
    "Inventory_Level"
]

ordinal_features = [
    "Age_Group"
]

nominal_features = [
    "Product_Category",
    "Product_Name",
    "Sales_Channel",
    "Customer_Segment",
    "Customer_Gender",
    "Country",
    "Shipping_Method",
    "Payment_Method",
    "Promotion_Code",
    "Order_Quarter",
    "Order_Year",
    "Day_of_Week"
]

features = (
    numeric_features
    + ordinal_features
    + nominal_features
)

leakage_columns = [
    "Profit",
    "Profit_Margin",
    "Sales_Amount",
    "Cost_Amount",
    "Gross_Sales",
    "Discount_Amount"
]

post_sale_columns = [
    "Return_Flag",
    "Return_Reason",
    "Is_Returned",
    "Delivery_Days",
    "Customer_Rating",
    "Order_Status"
]

print("Target:", target)
print("Número de predictores:", len(features))
print("Numéricas:", numeric_features)
print("Ordinal:", ordinal_features)
print("Nominales:", nominal_features)

Target: Profit_Margin
Número de predictores: 17
Numéricas: ['Discount_Percentage', 'Quantity', 'Unit_Price', 'Inventory_Level']
Ordinal: ['Age_Group']
Nominales: ['Product_Category', 'Product_Name', 'Sales_Channel', 'Customer_Segment', 'Customer_Gender', 'Country', 'Shipping_Method', 'Payment_Method', 'Promotion_Code', 'Order_Quarter', 'Order_Year', 'Day_of_Week']


## 4.5 Verificación automática de leakage

In [6]:
leakage_en_X = sorted(
    set(features).intersection(
        leakage_columns
    )
)

print(
    "Variables de leakage dentro de X:",
    leakage_en_X
)

assert len(leakage_en_X) == 0

print("Verificación superada.")

Variables de leakage dentro de X: []
Verificación superada.


# 5. Recuperar los valores faltantes originales

Durante el EDA algunos valores faltantes fueron tratados para poder analizar el dataset completo.

Para Machine Learning queremos que cualquier imputación se aprenda **solo con entrenamiento**.

Por eso:

1. usamos `Transaction_ID` para alinear el dataset limpio con el original;
2. restauramos los `NaN`;
3. después hacemos el `train_test_split`.

### Caso especial: `Promotion_Code`

Durante el EDA la ausencia de código se interpretó como **sin promoción**.

Por eso no restauramos esos valores como `NaN`: en este caso la ausencia tiene significado de negocio.

In [7]:
df_original_ml = (
    df_original
    .drop_duplicates()
    .copy()
)

if df["Transaction_ID"].duplicated().any():
    raise ValueError(
        "Transaction_ID está duplicado en el dataset limpio."
    )

if df_original_ml["Transaction_ID"].duplicated().any():
    raise ValueError(
        "Transaction_ID sigue duplicado en el dataset original."
    )

original_indexado = (
    df_original_ml
    .set_index("Transaction_ID")
)

for columna in features:

    if columna == "Promotion_Code":
        continue

    if columna in original_indexado.columns:

        ids_nan = original_indexado.index[
            original_indexado[columna].isna()
        ]

        mask = df["Transaction_ID"].isin(
            ids_nan
        )

        df.loc[
            mask,
            columna
        ] = np.nan


# Age_Group depende de Customer_Age
edad_original = original_indexado[
    "Customer_Age"
]

ids_edad_problematicos = edad_original.index[
    edad_original.isna()
    | (edad_original < 18)
    | (edad_original > 100)
]

mask_age = df["Transaction_ID"].isin(
    ids_edad_problematicos
)

df.loc[
    mask_age,
    "Age_Group"
] = np.nan

In [8]:
# Trimestre y año se tratarán como categorías,
# no como magnitudes numéricas.

for col in ["Order_Quarter", "Order_Year"]:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype("Int64")
            .astype("string")
        )


X = df[features].copy()
y = df[target].copy()

print("X:", X.shape)
print("y:", y.shape)

faltantes = pd.DataFrame({
    "NaN": X.isna().sum(),
    "% NaN": X.isna().mean() * 100
})

display(
    faltantes[
        faltantes["NaN"] > 0
    ]
    .sort_values(
        "% NaN",
        ascending=False
    )
    .round(2)
)

X: (18000, 17)
y: (18000,)


,NaN,% NaN
Age_Group,219,1.22
Inventory_Level,198,1.10
Customer_Gender,162,0.90
Product_Name,144,0.80
Payment_Method,144,0.80


# 6. Calidad financiera del target

Antes de entrenar revisamos una inconsistencia detectada durante el EDA:

`Profit` debería ser aproximadamente igual a:

`Sales_Amount - Cost_Amount`

No eliminaremos registros silenciosamente ni reemplazaremos el `Profit` reportado.

En cambio:

1. cuantificamos cuántos registros presentan discrepancias;
2. conservamos el target original;
3. después del modelado realizamos un **análisis de sensibilidad** para medir cuánto afectan estas anomalías al desempeño.

Usamos una tolerancia de **$0.02** para evitar marcar diferencias puramente de redondeo.

In [9]:
TOLERANCIA_PROFIT = 0.02

df[
    "Profit_Esperado"
] = (
    df["Sales_Amount"]
    - df["Cost_Amount"]
)

df[
    "Diferencia_Profit"
] = (
    df["Profit"]
    - df["Profit_Esperado"]
)

df[
    "Profit_Consistente"
] = (
    df["Diferencia_Profit"].abs()
    <= TOLERANCIA_PROFIT
)

n_inconsistentes = (
    (~df["Profit_Consistente"])
    .sum()
)

pct_inconsistentes = (
    n_inconsistentes
    / len(df)
    * 100
)

print(
    "Registros con inconsistencia:",
    n_inconsistentes
)

print(
    f"Porcentaje: {pct_inconsistentes:.2f}%"
)

display(
    df.loc[
        ~df["Profit_Consistente"],
        [
            "Transaction_ID",
            "Sales_Amount",
            "Cost_Amount",
            "Profit",
            "Profit_Esperado",
            "Diferencia_Profit",
            "Profit_Margin"
        ]
    ]
    .sort_values(
        "Diferencia_Profit",
        key=lambda s: s.abs(),
        ascending=False
    )
    .head(15)
    .round(2)
)

Registros con inconsistencia: 110
Porcentaje: 0.61%


,Transaction_ID,Sales_Amount,Cost_Amount,Profit,Profit_Esperado,Diferencia_Profit,Profit_Margin
14654,T02001097,2538.06,1853.19,363.30,684.87,-321.57,14.31
13050,T02016187,6932.33,5680.08,1553.94,1252.25,301.69,22.42
16315,T02012502,28.36,18.35,-280.34,10.01,-290.35,-988.50
10710,T02007752,291.26,236.11,-230.44,55.15,-285.59,-79.12
1857,T02000839,225.45,188.78,296.92,36.67,260.25,131.70
12880,T02002263,1034.86,968.33,291.57,66.53,225.04,28.17
13511,T02008620,1156.89,983.31,380.81,173.58,207.23,32.92
1985,T02002589,364.43,263.34,288.34,101.09,187.25,79.12
5819,T02010522,329.54,233.36,268.95,96.18,172.77,81.61
6541,T02000696,1068.64,994.75,245.89,73.89,172.00,23.01


### Decisión

Los registros inconsistentes **se conservan** porque el proyecto decidió trabajar con el `Profit` reportado.

Sin embargo, las métricas finales deben interpretarse junto con un análisis de sensibilidad sobre los registros financieramente consistentes. Así distinguimos entre limitaciones del modelo y posibles problemas de calidad del target.

# 7. División entrenamiento / prueba

Usaremos:

- 80 % entrenamiento;
- 20 % test final.

Todas las decisiones de:

- imputación;
- codificación;
- escalado;
- selección del modelo

se tomarán **solamente dentro de train**, utilizando validación cruzada.

`X_test` no participa en esas decisiones.

Esto evita que el desempeño final quede artificialmente optimista.

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (14400, 17)
Test : (3600, 17)


---
# ETAPA 1 — Imputación y Missing Indicator

Comparamos las tres estrategias principales vistas en clase:

1. `SimpleImputer`
2. `KNNImputer`
3. `SimpleImputer + MissingIndicator`

## ¿Por qué no elegir una automáticamente?

Porque una técnica más compleja solo tiene sentido si aporta una mejora útil.

Además, los faltantes pueden comportarse de formas distintas:

- puede bastar con una mediana;
- puede ser útil buscar observaciones similares;
- o el hecho de que un dato esté ausente puede contener señal.

In [11]:
missing_train = pd.DataFrame({
    "Cantidad_NaN": X_train.isna().sum(),
    "Porcentaje_NaN": X_train.isna().mean() * 100
})

missing_train = (
    missing_train[
        missing_train["Cantidad_NaN"] > 0
    ]
    .sort_values(
        "Porcentaje_NaN",
        ascending=False
    )
)

display(
    missing_train.round(2)
)

,Cantidad_NaN,Porcentaje_NaN
Age_Group,185,1.28
Inventory_Level,158,1.10
Customer_Gender,129,0.90
Payment_Method,118,0.82
Product_Name,109,0.76


## 7.1 Configuración auxiliar

Para comparar imputaciones necesitamos un modelo.

Usamos temporalmente:

- `StandardScaler`;
- `OneHotEncoder`;
- `OrdinalEncoder`;
- `Ridge`.

Estos elementos se mantienen constantes durante la comparación.

El objetivo aquí **no es escoger el modelo final**, sino aislar el efecto de la imputación.

In [12]:
orden_edad = [
    [
        "18-25",
        "26-35",
        "36-45",
        "46-55",
        "56+"
    ]
]


def crear_pipeline_imputacion(tipo):

    if tipo == "knn":

        num_pipe = Pipeline([
            (
                "imputer",
                KNNImputer(
                    n_neighbors=5,
                    weights="distance"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ])

    else:

        num_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ])

    ord_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "ordinal",
            OrdinalEncoder(
                categories=orden_edad,
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ])

    nom_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    transformers = [
        ("num", num_pipe, numeric_features),
        ("ord", ord_pipe, ordinal_features),
        ("nom", nom_pipe, nominal_features)
    ]

    if tipo == "indicator":

        transformers.append(
            (
                "missing",
                MissingIndicator(
                    features="missing-only",
                    error_on_new=False
                ),
                features
            )
        )

    prep = ColumnTransformer(
        transformers=transformers
    )

    return Pipeline([
        ("prep", prep),
        (
            "reg",
            Ridge(alpha=1.0)
        )
    ])

## 7.2 Validación cruzada de imputación

Para que `KNNImputer` no vuelva innecesariamente pesado el ejercicio, la comparación se hace sobre una muestra reproducible de hasta 6.000 filas tomada **solo de train**.

La métrica principal es:

### MAE

Interpretación:

> En promedio, ¿cuántos puntos porcentuales de margen se equivoca el modelo?

In [13]:
n_imp = min(
    6000,
    len(X_train)
)

X_imp = X_train.sample(
    n=n_imp,
    random_state=RANDOM_STATE
)

y_imp = y_train.loc[
    X_imp.index
]

cv_imp = KFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

estrategias_imp = {
    "simple": crear_pipeline_imputacion(
        "simple"
    ),
    "knn": crear_pipeline_imputacion(
        "knn"
    ),
    "indicator": crear_pipeline_imputacion(
        "indicator"
    )
}

resultados_imp = []

for clave, modelo in estrategias_imp.items():

    inicio = time.time()

    scores = cross_validate(
        modelo,
        X_imp,
        y_imp,
        cv=cv_imp,
        scoring={
            "mae": "neg_mean_absolute_error",
            "r2": "r2"
        },
        n_jobs=-1
    )

    resultados_imp.append({
        "Clave": clave,
        "Estrategia": {
            "simple": "SimpleImputer",
            "knn": "KNNImputer",
            "indicator": "Simple + MissingIndicator"
        }[clave],
        "MAE_CV": -scores["test_mae"].mean(),
        "MAE_std": scores["test_mae"].std(),
        "R2_CV": scores["test_r2"].mean(),
        "Tiempo_s": time.time() - inicio
    })


tabla_imputacion = (
    pd.DataFrame(resultados_imp)
    .sort_values(
        "MAE_CV"
    )
    .reset_index(drop=True)
)

display(
    tabla_imputacion.round(4)
)

,Clave,Estrategia,MAE_CV,MAE_std,R2_CV,Tiempo_s
0,indicator,Simple + MissingIndicator,2.3065,0.0440,0.5999,0.2665
1,knn,KNNImputer,2.3066,0.0433,0.5999,0.2540
2,simple,SimpleImputer,2.3066,0.0435,0.5999,3.3138


## 7.3 Decisión: desempeño + parsimonia

No queremos escoger KNN o indicadores por una mejora diminuta.

Regla:

- si una alternativa mejora el MAE de `SimpleImputer` en más de 1 %, la conservamos;
- si no, preferimos la estrategia sencilla.

Esta regla hace explícita la decisión y evita complejidad sin beneficio práctico.

In [14]:
mae_simple = (
    tabla_imputacion
    .loc[
        tabla_imputacion["Clave"] == "simple",
        "MAE_CV"
    ]
    .iloc[0]
)

mejor_fila_imp = (
    tabla_imputacion
    .iloc[0]
)

mejora_relativa_imp = (
    (mae_simple - mejor_fila_imp["MAE_CV"])
    / mae_simple
)

if (
    mejor_fila_imp["Clave"] != "simple"
    and mejora_relativa_imp > 0.01
):

    mejor_imputacion = (
        mejor_fila_imp["Clave"]
    )

else:

    mejor_imputacion = "simple"


print(
    "Mejor resultado observado:",
    mejor_fila_imp["Estrategia"]
)

print(
    "Mejora frente a SimpleImputer:",
    f"{mejora_relativa_imp * 100:.2f}%"
)

print(
    "Imputación acumulada:",
    mejor_imputacion
)

Mejor resultado observado: Simple + MissingIndicator
Mejora frente a SimpleImputer: 0.00%
Imputación acumulada: simple


---
# ETAPA 2 — Codificación categórica

Ahora conservamos la imputación seleccionada y añadimos la representación de las variables categóricas.

## Variables nominales → One-Hot Encoding

Ejemplos:

- producto;
- canal;
- país;
- método de pago.

No tienen un orden natural.

## Variable ordinal → Ordinal Encoding

`Age_Group` sí tiene un orden:

`18-25 < 26-35 < 36-45 < 46-55 < 56+`

## Trimestre y año

Aunque están escritos como números, los tratamos como categorías para no imponer una relación lineal artificial sobre el margen.

## 8.1 Ejemplo One-Hot

In [15]:
demo_ohe = (
    X_train[["Sales_Channel"]]
    .dropna()
    .head(8)
)

ohe_demo = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

ohe_array = ohe_demo.fit_transform(
    demo_ohe
)

display(
    pd.concat(
        [
            demo_ohe.reset_index(drop=True),
            pd.DataFrame(
                ohe_array,
                columns=ohe_demo.get_feature_names_out(
                    ["Sales_Channel"]
                )
            )
        ],
        axis=1
    )
)

,Sales_Channel,Sales_Channel_B2B Portal,Sales_Channel_Online,Sales_Channel_Retail Store
0,Online,0.0,1.0,0.0
1,Online,0.0,1.0,0.0
2,Online,0.0,1.0,0.0
3,B2B Portal,1.0,0.0,0.0
4,Online,0.0,1.0,0.0
5,Retail Store,0.0,0.0,1.0
6,Online,0.0,1.0,0.0
7,Retail Store,0.0,0.0,1.0


## 8.2 Ejemplo Ordinal

In [16]:
demo_age = (
    X_train[["Age_Group"]]
    .dropna()
    .head(8)
)

age_encoder = OrdinalEncoder(
    categories=orden_edad,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

age_cod = age_encoder.fit_transform(
    demo_age
)

display(
    pd.DataFrame({
        "Age_Group": demo_age["Age_Group"].values,
        "Age_Group_cod": age_cod.ravel()
    })
)

,Age_Group,Age_Group_cod
0,36-45,2.0
1,56+,4.0
2,26-35,1.0
3,46-55,3.0
4,18-25,0.0
5,46-55,3.0
6,26-35,1.0
7,26-35,1.0


## 8.3 Frequency Encoding

También mostramos la técnica vista en clase.

No la elegimos como estrategia principal porque tenemos una cardinalidad manejable y One-Hot conserva la identidad del producto, algo importante para estudiar rentabilidad.

In [17]:
freq_producto = (
    X_train["Product_Name"]
    .value_counts(
        normalize=True
    )
)

demo_freq = (
    X_train[["Product_Name"]]
    .head(10)
    .copy()
)

demo_freq["Frecuencia"] = (
    demo_freq["Product_Name"]
    .map(freq_producto)
)

display(demo_freq)

,Product_Name,Frecuencia
1921,NoiseCancel Headset,0.040445
5732,Webcam HD,0.041914
2455,Ergo Footrest,0.038206
12131,AeroBook 14 Laptop,0.041145
16612,Coffee Machine,0.022602
7136,Copy Paper Box,0.055350
2525,AeroBook 14 Laptop,0.041145
15931,SmartDock USB-C,0.035757
1684,Laser Toner,0.043104
14380,NoiseCancel Headset,0.040445


## 8.4 Preprocesador acumulativo

Esta función incorpora:

- la imputación de Etapa 1;
- la codificación de Etapa 2;
- y permitirá cambiar el escalador en Etapa 3.

In [18]:
def crear_numeric_pipe(
    imputacion,
    scaler=None
):

    if imputacion == "knn":

        imputer = KNNImputer(
            n_neighbors=5,
            weights="distance"
        )

    else:

        imputer = SimpleImputer(
            strategy="median"
        )

    pasos = [
        ("imputer", imputer)
    ]

    if scaler is not None:

        pasos.append(
            (
                "scaler",
                clone(scaler)
            )
        )

    return Pipeline(pasos)


def crear_preprocesador(
    imputacion,
    scaler=None,
    subset_features=None
):

    if subset_features is None:
        subset_features = features

    num_cols = [
        c for c in numeric_features
        if c in subset_features
    ]

    ord_cols = [
        c for c in ordinal_features
        if c in subset_features
    ]

    nom_cols = [
        c for c in nominal_features
        if c in subset_features
    ]

    transformers = []

    if len(num_cols) > 0:

        transformers.append(
            (
                "num",
                crear_numeric_pipe(
                    imputacion,
                    scaler
                ),
                num_cols
            )
        )

    if len(ord_cols) > 0:

        ord_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "ordinal",
                OrdinalEncoder(
                    categories=age_order,
                    handle_unknown="use_encoded_value",
                    unknown_value=-1
                )
            )
        ])

        transformers.append(
            (
                "ord",
                ord_pipe,
                ord_cols
            )
        )

    if len(nom_cols) > 0:

        nom_pipe = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ])

        transformers.append(
            (
                "nom",
                nom_pipe,
                nom_cols
            )
        )

    if imputacion == "indicator":

        indicadores = [
            c for c in subset_features
            if c in features
        ]

        if len(indicadores) > 0:

            transformers.append(
                (
                    "missing",
                    MissingIndicator(
                        features="missing-only",
                        error_on_new=False
                    ),
                    indicadores
                )
            )

    return ColumnTransformer(
        transformers=transformers
    )

## 8.5 Contraste didáctico: todo como ordinal

Este experimento demuestra por qué una representación numérica arbitraria puede ser engañosa.

Aunque las métricas fueran parecidas, convertir `Online`, `Retail` y `B2B` en 0, 1 y 2 crea una jerarquía que no existe.

La codificación correcta se conserva por coherencia conceptual.

In [19]:
def crear_preprocesador_todo_ordinal(
    imputacion,
    scaler=None
):

    num_pipe = crear_numeric_pipe(
        imputacion,
        scaler
    )

    cat_pipe = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "ordinal",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            )
        )
    ])

    transformers = [
        ("num", num_pipe, numeric_features),
        (
            "cat",
            cat_pipe,
            ordinal_features + nominal_features
        )
    ]

    if imputacion == "indicator":

        transformers.append(
            (
                "missing",
                MissingIndicator(
                    features="missing-only",
                    error_on_new=False
                ),
                features
            )
        )

    return ColumnTransformer(
        transformers=transformers
    )

In [20]:
cv_pre = KFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

codificaciones = {
    "Codificación correcta":
        crear_preprocesador(
            mejor_imputacion,
            StandardScaler()
        ),

    "Todo como ordinal":
        crear_preprocesador_todo_ordinal(
            mejor_imputacion,
            StandardScaler()
        )
}

res_cod = []

for nombre, prep in codificaciones.items():

    modelo = Pipeline([
        ("prep", prep),
        ("reg", Ridge(alpha=1.0))
    ])

    scores = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=cv_pre,
        scoring={
            "mae": "neg_mean_absolute_error",
            "r2": "r2"
        },
        n_jobs=-1
    )

    res_cod.append({
        "Estrategia": nombre,
        "MAE_CV": -scores["test_mae"].mean(),
        "R2_CV": scores["test_r2"].mean()
    })


tabla_codificacion = pd.DataFrame(
    res_cod
)

display(
    tabla_codificacion.round(4)
)

NameError: name 'age_order' is not defined

### Decisión de Etapa 2

Se conserva:

- One-Hot para nominales;
- OrdinalEncoder para `Age_Group`.

No elegimos una codificación únicamente porque una métrica cambie unas décimas: la representación debe respetar el significado del dato.

---
# ETAPA 3 — Escalado y normalización

Nuestras numéricas viven en rangos muy distintos.

Ejemplos:

- `Unit_Price` puede estar en cientos o miles;
- `Discount_Percentage` es un porcentaje;
- `Quantity` se mueve en unidades pequeñas.

Los modelos lineales regularizados y los modelos basados en distancia pueden verse afectados por esas diferencias.

Comparamos:

- sin escalado;
- `StandardScaler`;
- `MinMaxScaler`;
- `RobustScaler`;
- `Normalizer`.

In [ ]:
resumen_num = pd.DataFrame({
    "min": X_train[numeric_features].min(),
    "max": X_train[numeric_features].max(),
    "media": X_train[numeric_features].mean(),
    "mediana": X_train[numeric_features].median(),
    "std": X_train[numeric_features].std()
})

display(
    resumen_num.round(2)
)

## 9.1 Interpretación de cada alternativa

### StandardScaler
Expresa cada variable aproximadamente en unidades de desviación estándar.

### MinMaxScaler
Lleva cada variable a un rango común.

### RobustScaler
Utiliza mediana e IQR; es menos sensible a valores extremos.

### Normalizer
Normaliza cada **fila completa**.

Lo estudiamos porque forma parte del material de clase, pero no es un candidato natural para nuestro modelo: queremos preservar el significado de magnitudes como precio y descuento.

In [ ]:
scalers = {
    "Sin escalado": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
    "Normalizer L2": Normalizer(norm="l2")
}

res_scaler = []

for nombre, scaler in scalers.items():

    inicio = time.time()

    modelo = Pipeline([
        (
            "prep",
            crear_preprocesador(
                mejor_imputacion,
                scaler
            )
        ),
        (
            "reg",
            Ridge(alpha=1.0)
        )
    ])

    scores = cross_validate(
        modelo,
        X_train,
        y_train,
        cv=cv_pre,
        scoring={
            "mae": "neg_mean_absolute_error",
            "r2": "r2"
        },
        n_jobs=-1
    )

    res_scaler.append({
        "Escalado": nombre,
        "MAE_CV": -scores["test_mae"].mean(),
        "MAE_std": scores["test_mae"].std(),
        "R2_CV": scores["test_r2"].mean(),
        "Tiempo_s": time.time() - inicio
    })


tabla_scaler = (
    pd.DataFrame(res_scaler)
    .sort_values("MAE_CV")
    .reset_index(drop=True)
)

display(
    tabla_scaler.round(4)
)

## 9.2 Decisión de escalado

Usamos el mismo principio de parsimonia:

- `Normalizer` se excluye de la decisión final por su interpretación por fila;
- si otra alternativa mejora a `StandardScaler` por más de 1 %, la conservamos;
- si no, mantenemos `StandardScaler`.

Esto evita convertir el escalado en un dogma: lo evaluamos, pero también justificamos su significado.

In [ ]:
candidatos_scaler = tabla_scaler[
    tabla_scaler["Escalado"] != "Normalizer L2"
].copy()

mejor_fila_scaler = (
    candidatos_scaler
    .sort_values("MAE_CV")
    .iloc[0]
)

mae_standard = (
    candidatos_scaler
    .loc[
        candidatos_scaler["Escalado"] == "StandardScaler",
        "MAE_CV"
    ]
    .iloc[0]
)

mejora_vs_standard = (
    (mae_standard - mejor_fila_scaler["MAE_CV"])
    / mae_standard
)

if (
    mejor_fila_scaler["Escalado"] != "StandardScaler"
    and mejora_vs_standard > 0.01
):

    mejor_scaler_nombre = (
        mejor_fila_scaler["Escalado"]
    )

else:

    mejor_scaler_nombre = "StandardScaler"


mapa_scalers = {
    "Sin escalado": None,
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler()
}

mejor_scaler = mapa_scalers[
    mejor_scaler_nombre
]

print(
    "Mejor observado:",
    mejor_fila_scaler["Escalado"]
)

print(
    "Mejora frente a StandardScaler:",
    f"{mejora_vs_standard * 100:.2f}%"
)

print(
    "Escalado acumulado:",
    mejor_scaler_nombre
)

---
# ETAPA 4 — Texto y vectorización

El material de clase trabaja:

- `CountVectorizer`;
- stopwords;
- n-gramas;
- `TF-IDF`.

Pero antes de aplicar una técnica debemos determinar si existe **texto libre**.

No toda columna de tipo `object` es texto lingüístico.

In [ ]:
posibles_textos = [
    col
    for col in [
        "Customer_Name",
        "Product_Name",
        "Return_Reason"
    ]
    if col in df.columns
]

resumen_texto = []

for col in posibles_textos:

    serie = (
        df[col]
        .dropna()
        .astype(str)
    )

    resumen_texto.append({
        "Variable": col,
        "Valores_unicos": serie.nunique(),
        "Promedio_caracteres": serie.str.len().mean(),
        "Ejemplo": (
            serie.iloc[0]
            if len(serie) > 0
            else ""
        )
    })

display(
    pd.DataFrame(resumen_texto)
)

## 10.1 Decisión sobre texto

### Customer_Name
No se utiliza: es un identificador.

### Return_Reason
No se utiliza: ocurre después de la venta.

### Product_Name
Sí se utiliza en el modelo, pero como **categoría nominal**.

**¿Por qué no TF-IDF?**

Porque nombres de productos como `"AeroBook 14 Laptop"` funcionan principalmente como etiquetas de catálogo. No tenemos descripciones, comentarios o reseñas donde el significado de las palabras sea el objeto del análisis.

Aun así, realizamos un experimento didáctico con CountVectorizer y TF-IDF para demostrar la técnica.

In [ ]:
product_text_train = (
    X_train["Product_Name"]
    .fillna("")
    .astype(str)
)

count_vec = CountVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_count = count_vec.fit_transform(
    product_text_train
)

print(
    "CountVectorizer:",
    X_count.shape
)

print(
    count_vec
    .get_feature_names_out()[:30]
)

In [ ]:
tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2
)

X_tfidf = tfidf_vec.fit_transform(
    product_text_train
)

print(
    "TF-IDF:",
    X_tfidf.shape
)

print(
    tfidf_vec
    .get_feature_names_out()[:30]
)

### Decisión de Etapa 4

La vectorización se estudia, pero no entra al pipeline principal.

Esto también es una conclusión válida:

> **Una técnica no debe aplicarse solo para marcarla como completada; debe corresponder al tipo de variable disponible.**

# 11. Pipeline acumulativo después de las cuatro etapas

In [ ]:
preprocesador_final = crear_preprocesador(
    mejor_imputacion,
    mejor_scaler
)

X_train_preparado = (
    preprocesador_final
    .fit_transform(X_train)
)

print(
    "Variables originales:",
    X_train.shape[1]
)

print(
    "Variables después del preprocesamiento:",
    X_train_preparado.shape[1]
)

print(
    "\nEl test sigue reservado."
)

---
# 12. Comparación de modelos de regresión

Ahora sí comparamos modelos.

## Baseline

Primero incluimos:

`DummyRegressor`

Este modelo predice siempre una constante.

Sirve para responder:

> ¿Nuestros predictores realmente contienen información o un modelo ingenuo obtiene algo parecido?

## Modelos candidatos

Comparamos:

- regresión lineal;
- Ridge;
- árbol;
- KNN;
- Random Forest;
- Gradient Boosting.

## Métrica principal: MAE

El MAE es especialmente útil porque se interpreta directamente como:

> **error promedio en puntos porcentuales de margen**

También reportamos RMSE, R², variabilidad entre folds y tiempo.

In [ ]:
modelos = {
    "Dummy (media)": DummyRegressor(
        strategy="mean"
    ),

    "Regresión lineal": LinearRegression(),

    "Ridge": Ridge(
        alpha=1.0
    ),

    "Árbol de decisión": DecisionTreeRegressor(
        max_depth=8,
        min_samples_leaf=20,
        random_state=RANDOM_STATE
    ),

    "KNN": KNeighborsRegressor(
        n_neighbors=25,
        weights="distance"
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),

    "Gradient Boosting": HistGradientBoostingRegressor(
        learning_rate=0.08,
        max_iter=250,
        random_state=RANDOM_STATE
    )
}

In [ ]:
cv_modelos = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

resultados_modelos = []
pipelines_modelos = {}

for nombre, reg in modelos.items():

    print("Evaluando:", nombre)

    pipe = Pipeline([
        (
            "prep",
            crear_preprocesador(
                mejor_imputacion,
                mejor_scaler
            )
        ),
        (
            "reg",
            reg
        )
    ])

    pipelines_modelos[nombre] = pipe

    inicio = time.time()

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv_modelos,
        scoring={
            "mae": "neg_mean_absolute_error",
            "rmse": "neg_root_mean_squared_error",
            "r2": "r2"
        },
        n_jobs=-1
    )

    resultados_modelos.append({
        "Modelo": nombre,
        "MAE_CV": -scores["test_mae"].mean(),
        "MAE_std": scores["test_mae"].std(),
        "RMSE_CV": -scores["test_rmse"].mean(),
        "R2_CV": scores["test_r2"].mean(),
        "R2_std": scores["test_r2"].std(),
        "Tiempo_s": time.time() - inicio
    })


tabla_modelos = (
    pd.DataFrame(resultados_modelos)
    .sort_values("MAE_CV")
    .reset_index(drop=True)
)

display(
    tabla_modelos.round(4)
)

# 13. Selección del modelo con principio de parsimonia

No escogemos automáticamente el primer lugar si la mejora es mínima.

Procedimiento:

1. excluimos el Dummy de la selección;
2. identificamos el menor MAE;
3. consideramos como prácticamente empatados los modelos cuyo MAE esté a máximo 1 % del mejor;
4. dentro de ese grupo preferimos el modelo más sencillo e interpretable.

Esto evita pagar complejidad por una diferencia irrelevante.

In [ ]:
tabla_reales = tabla_modelos[
    tabla_modelos["Modelo"] != "Dummy (media)"
].copy()

mejor_mae = (
    tabla_reales["MAE_CV"]
    .min()
)

candidatos_empatados = tabla_reales[
    tabla_reales["MAE_CV"]
    <= mejor_mae * 1.01
].copy()

prioridad_simplicidad = {
    "Regresión lineal": 1,
    "Ridge": 2,
    "Árbol de decisión": 3,
    "KNN": 4,
    "Gradient Boosting": 5,
    "Random Forest": 6
}

candidatos_empatados[
    "Prioridad"
] = candidatos_empatados[
    "Modelo"
].map(
    prioridad_simplicidad
)

candidatos_empatados = (
    candidatos_empatados
    .sort_values(
        [
            "Prioridad",
            "MAE_CV"
        ]
    )
)

mejor_nombre = (
    candidatos_empatados
    .iloc[0]["Modelo"]
)

mejor_modelo = pipelines_modelos[
    mejor_nombre
]

print(
    "Menor MAE absoluto:",
    tabla_reales
    .sort_values("MAE_CV")
    .iloc[0]["Modelo"]
)

print(
    "Modelo seleccionado con parsimonia:",
    mejor_nombre
)

display(
    candidatos_empatados[
        [
            "Modelo",
            "MAE_CV",
            "RMSE_CV",
            "R2_CV",
            "Tiempo_s"
        ]
    ].round(4)
)

# 14. Evaluación final en test

Ahora, y solo ahora, utilizamos `X_test`.

El modelo seleccionado se entrena con todo `X_train` y se evalúa una sola vez sobre el test reservado.

In [ ]:
mejor_modelo.fit(
    X_train,
    y_train
)

pred_test = mejor_modelo.predict(
    X_test
)

mae_test = mean_absolute_error(
    y_test,
    pred_test
)

rmse_test = np.sqrt(
    mean_squared_error(
        y_test,
        pred_test
    )
)

r2_test = r2_score(
    y_test,
    pred_test
)

print(
    "Modelo final:",
    mejor_nombre
)

print(
    f"MAE  test: {mae_test:.4f}"
)

print(
    f"RMSE test: {rmse_test:.4f}"
)

print(
    f"R²   test: {r2_test:.4f}"
)

## 14.1 Comparación contra el baseline

Un buen modelo debe mejorar claramente al Dummy.

In [ ]:
dummy_final = Pipeline([
    (
        "prep",
        crear_preprocesador(
            mejor_imputacion,
            mejor_scaler
        )
    ),
    (
        "reg",
        DummyRegressor(
            strategy="mean"
        )
    )
])

dummy_final.fit(
    X_train,
    y_train
)

pred_dummy = dummy_final.predict(
    X_test
)

mae_dummy = mean_absolute_error(
    y_test,
    pred_dummy
)

mejora_dummy = (
    (mae_dummy - mae_test)
    / mae_dummy
    * 100
)

print(
    f"MAE Dummy : {mae_dummy:.4f}"
)

print(
    f"MAE modelo: {mae_test:.4f}"
)

print(
    f"Mejora frente al Dummy: {mejora_dummy:.2f}%"
)

## 14.2 Real vs. predicho

In [ ]:
plt.figure(
    figsize=(7, 6)
)

plt.scatter(
    y_test,
    pred_test,
    alpha=0.30
)

lim_min = min(
    y_test.min(),
    pred_test.min()
)

lim_max = max(
    y_test.max(),
    pred_test.max()
)

plt.plot(
    [lim_min, lim_max],
    [lim_min, lim_max],
    linestyle="--"
)

plt.xlabel(
    "Profit_Margin real (%)"
)

plt.ylabel(
    "Profit_Margin predicho (%)"
)

plt.title(
    "Margen real vs. margen predicho"
)

plt.tight_layout()
plt.show()

## 14.3 Distribución de errores

In [ ]:
residuos = (
    y_test.values
    - pred_test
)

plt.figure(
    figsize=(8, 5)
)

plt.hist(
    residuos,
    bins=40
)

plt.axvline(
    0,
    linestyle="--"
)

plt.xlabel(
    "Error = margen real - margen predicho"
)

plt.ylabel(
    "Frecuencia"
)

plt.title(
    "Distribución de errores"
)

plt.tight_layout()
plt.show()

# 16. Análisis de sensibilidad por inconsistencias financieras

El MAE, RMSE y R² del test mezclan dos fenómenos:

- el error real del modelo;
- y posibles errores o inconsistencias del dato financiero usado como target.

Por eso comparamos el desempeño en:

1. todo el test;
2. registros financieramente consistentes;
3. registros inconsistentes.

> **Importante:** no usamos esta comparación para borrar observaciones después de ver el test. Es un diagnóstico de calidad de datos.

In [ ]:
calidad_test = (
    df.loc[
        X_test.index,
        [
            "Transaction_ID",
            "Profit_Consistente",
            "Diferencia_Profit"
        ]
    ]
    .copy()
)

calidad_test[
    "Profit_Margin_real"
] = y_test.values

calidad_test[
    "Profit_Margin_pred"
] = pred_test

calidad_test[
    "Error_Absoluto"
] = np.abs(
    calidad_test["Profit_Margin_real"]
    - calidad_test["Profit_Margin_pred"]
)

print(
    "Filas consistentes en test:",
    calidad_test[
        "Profit_Consistente"
    ].sum()
)

print(
    "Filas inconsistentes en test:",
    (
        ~calidad_test[
            "Profit_Consistente"
        ]
    ).sum()
)

In [ ]:
resultados_calidad = []

for nombre, mask in {
    "Test completo":
        np.ones(
            len(calidad_test),
            dtype=bool
        ),

    "Solo registros consistentes":
        calidad_test[
            "Profit_Consistente"
        ].values,

    "Solo registros inconsistentes":
        (
            ~calidad_test[
                "Profit_Consistente"
            ]
        ).values
}.items():

    y_real_grupo = calidad_test.loc[
        mask,
        "Profit_Margin_real"
    ]

    y_pred_grupo = calidad_test.loc[
        mask,
        "Profit_Margin_pred"
    ]

    if len(y_real_grupo) == 0:
        continue

    fila = {
        "Grupo": nombre,
        "N": len(y_real_grupo),
        "MAE": mean_absolute_error(
            y_real_grupo,
            y_pred_grupo
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                y_real_grupo,
                y_pred_grupo
            )
        )
    }

    if len(y_real_grupo) >= 2:

        fila["R2"] = r2_score(
            y_real_grupo,
            y_pred_grupo
        )

    else:

        fila["R2"] = np.nan

    resultados_calidad.append(
        fila
    )


tabla_calidad = pd.DataFrame(
    resultados_calidad
)

display(
    tabla_calidad.round(4)
)

## Cómo interpretar esta tabla

Si los registros inconsistentes presentan errores mucho mayores, una parte de la pérdida de desempeño no proviene necesariamente de que el modelo no haya aprendido el patrón, sino de que el target contiene observaciones difíciles de reconciliar financieramente.

Esta diferencia debe reportarse como **limitación de calidad de datos**, no como una excusa para eliminar casos después de conocer el resultado.

# 17. ¿Demasiado bueno para ser verdad?

Si obtenemos un R² extraordinariamente alto o un MAE muy pequeño, no debemos celebrarlo inmediatamente.

Primero revisamos dos posibilidades:

1. fuga de información;
2. una estructura del negocio que haga el target altamente predecible.

Ya verificamos que `X` no contiene las variables que construyen directamente el margen.

Ahora usamos `Cost_Amount` y `Gross_Sales` **solo como diagnóstico posterior**, nunca como predictores.

La pregunta es:

> ¿Dentro de cada producto el costo relativo es casi constante?

Si lo es, entonces producto + descuento pueden contener una señal muy fuerte sobre el margen.

In [ ]:
print(
    "Leakage dentro de X:",
    sorted(
        set(features).intersection(
            leakage_columns
        )
    )
)

if {
    "Cost_Amount",
    "Gross_Sales",
    "Product_Name"
}.issubset(df.columns):

    diagnostico = df.copy()

    diagnostico[
        "Costo_Relativo"
    ] = np.where(
        diagnostico["Gross_Sales"].ne(0),
        diagnostico["Cost_Amount"]
        / diagnostico["Gross_Sales"],
        np.nan
    )

    estructura_producto = (
        diagnostico
        .groupby("Product_Name")[
            "Costo_Relativo"
        ]
        .agg(
            ["mean", "std", "count"]
        )
        .sort_values("std")
    )

    print(
        "\nDesviación estándar mediana "
        "del costo relativo por producto:",
        round(
            estructura_producto[
                "std"
            ].median(),
            4
        )
    )

    display(
        estructura_producto.head(30).round(4)
    )

# 18. Importancia por permutación

La pregunta de negocio exige algo más que precisión.

Queremos saber:

> **¿Qué variables pierden más capacidad predictiva cuando las desordenamos?**

La importancia por permutación:

1. calcula el desempeño normal;
2. desordena una variable;
3. vuelve a medir el MAE;
4. si el MAE aumenta mucho, esa variable era importante.

Esto funciona sobre las **variables originales**, antes del One-Hot, y es mucho más fácil de interpretar para negocio.

In [ ]:
perm = permutation_importance(
    mejor_modelo,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=8,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importancias = pd.DataFrame({
    "Variable": X_test.columns,
    "Aumento_MAE": perm.importances_mean,
    "Desviacion": perm.importances_std
})

importancias = (
    importancias
    .sort_values(
        "Aumento_MAE",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    importancias.round(4)
)

In [ ]:
plt.figure(
    figsize=(8, 7)
)

imp_plot = (
    importancias
    .sort_values(
        "Aumento_MAE"
    )
)

plt.barh(
    imp_plot["Variable"],
    imp_plot["Aumento_MAE"]
)

plt.xlabel(
    "Aumento del MAE al permutar"
)

plt.title(
    "Importancia predictiva de las variables"
)

plt.tight_layout()
plt.show()

# 19. Respuesta por dimensiones de negocio

Agrupamos las variables para volver a la pregunta original.

### Condiciones de la venta
Descuento, cantidad, precio e inventario.

### Producto
Nombre y categoría.

### Cliente
Segmento, edad, género y país.

### Canales
Canal, pago, envío y promoción.

### Periodos
Trimestre, año y día.

Las pequeñas importancias negativas producidas por ruido estadístico se llevan a cero antes de sumar.

In [ ]:
imp_map = (
    importancias
    .set_index("Variable")[
        "Aumento_MAE"
    ]
)

def importancia_dimension(lista):

    valores = [
        max(
            0,
            float(
                imp_map.get(
                    v,
                    0
                )
            )
        )
        for v in lista
    ]

    return sum(valores)


dimensiones = {
    "Condiciones de la venta": [
        "Discount_Percentage",
        "Quantity",
        "Unit_Price",
        "Inventory_Level"
    ],

    "Producto": [
        "Product_Name",
        "Product_Category"
    ],

    "Cliente": [
        "Customer_Segment",
        "Age_Group",
        "Customer_Gender",
        "Country"
    ],

    "Canales": [
        "Sales_Channel",
        "Payment_Method",
        "Shipping_Method",
        "Promotion_Code"
    ],

    "Periodos": [
        "Order_Quarter",
        "Order_Year",
        "Day_of_Week"
    ]
}


tabla_dimensiones = pd.DataFrame({
    "Dimension": dimensiones.keys(),
    "Importancia_Agregada": [
        importancia_dimension(
            variables
        )
        for variables in dimensiones.values()
    ]
})

tabla_dimensiones = (
    tabla_dimensiones
    .sort_values(
        "Importancia_Agregada",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    tabla_dimensiones.round(4)
)

In [ ]:
plt.figure(
    figsize=(8, 4.5)
)

plot_dim = (
    tabla_dimensiones
    .sort_values(
        "Importancia_Agregada"
    )
)

plt.barh(
    plot_dim["Dimension"],
    plot_dim["Importancia_Agregada"]
)

plt.xlabel(
    "Importancia agregada"
)

plt.title(
    "¿Qué dimensiones aportan más a explicar la rentabilidad?"
)

plt.tight_layout()
plt.show()

## Cómo interpretar esta tabla

Si `Periodos` pierde importancia una vez que el modelo conoce producto y descuento, eso sugiere que el trimestre puede estar funcionando principalmente como **contexto donde cambian las políticas comerciales**, no necesariamente como un factor independiente.

Si `Cliente` o `Canales` tienen baja importancia, eso no significa que no sean importantes para ventas.

Puede significar que explican **volumen**, pero aportan poco para explicar la **eficiencia de margen**.

> Nuevamente: esto describe asociación predictiva, no causalidad.

# 20. ¿Necesitamos todas las variables?

La importancia por permutación puede sugerir que gran parte de la señal se concentra en pocas variables.

Para comprobarlo **sin usar el test para seleccionar variables**, hacemos un experimento adicional mediante validación cruzada solamente sobre `X_train`.

Comparamos:

1. todas las variables;
2. producto + descuento + precio + categoría;
3. producto + descuento;
4. todas menos `Product_Name`;
5. solo descuento.

Usamos el mismo algoritmo que fue seleccionado previamente. De esta manera cambia principalmente la información disponible, no la familia del modelo.

In [ ]:
feature_sets = {
    "Todas las variables":
        features,

    "Producto + descuento + precio + categoría": [
        "Product_Name",
        "Discount_Percentage",
        "Unit_Price",
        "Product_Category"
    ],

    "Producto + descuento": [
        "Product_Name",
        "Discount_Percentage"
    ],

    "Sin Product_Name": [
        c for c in features
        if c != "Product_Name"
    ],

    "Solo descuento": [
        "Discount_Percentage"
    ]
}

resultados_features = []

for nombre_set, cols in feature_sets.items():

    prep_subset = crear_preprocesador(
        mejor_imputacion,
        mejor_scaler,
        subset_features=cols
    )

    modelo_subset = Pipeline([
        ("prep", prep_subset),
        (
            "reg",
            clone(
                modelos[
                    mejor_nombre
                ]
            )
        )
    ])

    scores = cross_validate(
        modelo_subset,
        X_train[cols],
        y_train,
        cv=cv_modelos,
        scoring={
            "mae": "neg_mean_absolute_error",
            "r2": "r2"
        },
        n_jobs=1
    )

    resultados_features.append({
        "Conjunto": nombre_set,
        "N_variables_originales": len(cols),
        "MAE_CV": -scores["test_mae"].mean(),
        "R2_CV": scores["test_r2"].mean()
    })


tabla_features = (
    pd.DataFrame(
        resultados_features
    )
    .sort_values("MAE_CV")
    .reset_index(drop=True)
)

display(
    tabla_features.round(4)
)

## Cómo interpretar esta comparación

Si un conjunto pequeño obtiene un MAE muy parecido al modelo completo, podemos concluir que gran parte de la **señal predictiva del margen** se concentra en esas variables.

Esto no significa que las demás variables sean inútiles para el negocio. Pueden seguir siendo importantes para volumen, segmentación, campañas u operaciones, aunque aporten poco para explicar la eficiencia de margen.

# 21. Análisis de errores por segmentos

Un MAE global puede ocultar zonas problemáticas.

Revisamos si el modelo se equivoca más por:

- trimestre;
- categoría;
- producto;
- canal.

Esto ayuda a detectar segmentos donde hacen falta mejores datos o donde el comportamiento es más difícil de modelar.

In [ ]:
evaluacion = X_test.copy()

evaluacion[
    "Profit_Margin_real"
] = y_test.values

evaluacion[
    "Profit_Margin_pred"
] = pred_test

evaluacion[
    "Error_Absoluto"
] = np.abs(
    evaluacion[
        "Profit_Margin_real"
    ]
    - evaluacion[
        "Profit_Margin_pred"
    ]
)


for variable in [
    "Order_Quarter",
    "Product_Category",
    "Product_Name",
    "Sales_Channel"
]:

    print(
        "\nERROR POR:",
        variable
    )

    tabla_error = (
        evaluacion
        .groupby(variable)[
            "Error_Absoluto"
        ]
        .agg(
            ["mean", "median", "count"]
        )
        .sort_values(
            "mean",
            ascending=False
        )
    )

    display(
        tabla_error.head(15).round(3)
    )

# 22. Aplicación de negocio: alerta de baja rentabilidad

Mantenemos la regresión como problema principal.

Después traducimos el margen predicho a una regla de negocio.

Ejemplo:

`margen predicho < 15 % → alerta`

### Ventaja

Si mañana el negocio cambia 15 % por 12 % o 18 %, no necesitamos redefinir ni reentrenar el problema como clasificación.

El umbral de 15 % se usa como ejemplo y debe ser validado con el área financiera.

In [ ]:
UMBRAL_MARGEN = 15.0

alerta_real = (
    y_test < UMBRAL_MARGEN
).astype(int)

alerta_pred = (
    pred_test < UMBRAL_MARGEN
).astype(int)

precision_alerta = precision_score(
    alerta_real,
    alerta_pred,
    zero_division=0
)

recall_alerta = recall_score(
    alerta_real,
    alerta_pred,
    zero_division=0
)

f1_alerta = f1_score(
    alerta_real,
    alerta_pred,
    zero_division=0
)

print(
    f"Umbral operativo: {UMBRAL_MARGEN}%"
)

print(
    f"Precision: {precision_alerta:.4f}"
)

print(
    f"Recall   : {recall_alerta:.4f}"
)

print(
    f"F1       : {f1_alerta:.4f}"
)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    alerta_real,
    alerta_pred,
    display_labels=[
        "Margen normal",
        "Baja rentabilidad"
    ]
)

plt.title(
    f"Alerta derivada de la regresión (< {UMBRAL_MARGEN}%)"
)

plt.tight_layout()
plt.show()

# 23. Simulación predictiva: descuento por producto

Construimos una transacción representativa y modificamos el descuento para varios productos.

Para cada producto utilizamos su categoría y precio típico, mientras mantenemos valores representativos para las demás variables.

Después observamos cómo cambia el margen predicho.

> Esta es una **simulación predictiva**, no una estimación causal.

El gráfico permite comparar qué tan cerca queda cada producto del umbral de rentabilidad al variar el descuento, pero la interpretación depende del tipo de modelo seleccionado.

In [ ]:
fila_base = {}

for col in numeric_features:

    fila_base[col] = (
        X_train[col]
        .median()
    )

for col in ordinal_features + nominal_features:

    fila_base[col] = (
        X_train[col]
        .mode(dropna=True)
        .iloc[0]
    )


productos_demo = (
    X_train["Product_Name"]
    .value_counts()
    .head(6)
    .index
    .tolist()
)

descuentos_demo = np.arange(
    0,
    31,
    5
)

simulaciones = []

for producto in productos_demo:

    datos_producto = X_train[
        X_train["Product_Name"]
        == producto
    ]

    if len(datos_producto) == 0:
        continue

    precio_tipico = (
        datos_producto[
            "Unit_Price"
        ]
        .median()
    )

    categoria_tipica = (
        datos_producto[
            "Product_Category"
        ]
        .mode()
        .iloc[0]
    )

    for descuento in descuentos_demo:

        fila = fila_base.copy()

        fila.update({
            "Product_Name": producto,
            "Product_Category": categoria_tipica,
            "Unit_Price": precio_tipico,
            "Discount_Percentage": float(descuento)
        })

        fila_df = pd.DataFrame(
            [fila]
        )[features]

        margen_pred = (
            mejor_modelo
            .predict(
                fila_df
            )[0]
        )

        simulaciones.append({
            "Product_Name": producto,
            "Discount_Percentage": descuento,
            "Profit_Margin_pred": margen_pred
        })


df_simulacion = pd.DataFrame(
    simulaciones
)

display(
    df_simulacion.head(20).round(3)
)

In [ ]:
plt.figure(
    figsize=(10, 6)
)

for producto in productos_demo:

    sub = df_simulacion[
        df_simulacion[
            "Product_Name"
        ] == producto
    ]

    plt.plot(
        sub[
            "Discount_Percentage"
        ],
        sub[
            "Profit_Margin_pred"
        ],
        marker="o",
        label=producto
    )

plt.axhline(
    UMBRAL_MARGEN,
    linestyle="--",
    label=f"Umbral {UMBRAL_MARGEN}%"
)

plt.xlabel(
    "Descuento aplicado (%)"
)

plt.ylabel(
    "Margen predicho (%)"
)

plt.title(
    "Simulación predictiva: descuento y margen por producto"
)

plt.legend()

plt.tight_layout()
plt.show()

## 23.1 Limitación clave: producto × descuento

Si el modelo seleccionado es **Regresión Lineal o Ridge**, no hemos incluido explícitamente una interacción:

`Product_Name × Discount_Percentage`

Eso significa que cada producto puede tener un margen base diferente, pero el efecto lineal del descuento es esencialmente común entre productos.

Por lo tanto, **no debemos afirmar que el modelo demostró elasticidades o sensibilidades distintas al descuento para cada producto**.

La conclusión correcta es:

> Debido a sus distintos márgenes base, diferentes productos pueden alcanzar el umbral de rentabilidad con niveles distintos de descuento según la predicción del modelo.

Para estudiar sensibilidad específica por producto haría falta incluir interacciones explícitas o utilizar un diseño analítico orientado a ese objetivo.

In [ ]:
if mejor_nombre in [
    "Regresión lineal",
    "Ridge"
]:

    print(
        "Modelo lineal seleccionado."
    )

    print(
        "La simulación no incluye una interacción "
        "explícita Producto × Descuento."
    )

else:

    print(
        "Modelo no lineal seleccionado."
    )

    print(
        "Puede capturar algunas interacciones de forma implícita, "
        "pero la simulación sigue siendo predictiva y no causal."
    )

## 23.2 Mayor descuento probado que conserva el margen de referencia

Dentro de los niveles simulados buscamos el mayor descuento que mantiene:

`Profit_Margin_pred >= 15 %`

Este resultado es **orientativo**. No demuestra una elasticidad propia del producto ni debe convertirse automáticamente en política comercial.

In [ ]:
topes = []

for producto in productos_demo:

    sub = (
        df_simulacion[
            df_simulacion[
                "Product_Name"
            ] == producto
        ]
        .sort_values(
            "Discount_Percentage"
        )
    )

    seguros = sub[
        sub[
            "Profit_Margin_pred"
        ] >= UMBRAL_MARGEN
    ]

    if len(seguros) > 0:

        tope = seguros[
            "Discount_Percentage"
        ].max()

    else:

        tope = np.nan

    topes.append({
        "Product_Name": producto,
        "Mayor_descuento_probado_con_margen_seguro": tope
    })


tabla_topes = pd.DataFrame(
    topes
)

display(tabla_topes)

# 24. Conclusión metodológica

Este notebook responde la pregunta original sin reducir prematuramente la rentabilidad a una etiqueta.

La lógica final es:

### EDA
Identifica asociaciones descriptivas.

### Cuatro etapas de preprocesamiento
Justifican cómo se preparan los datos y por qué se conserva o descarta cada técnica.

### Comparación de regresores
Determina qué modelo generaliza mejor sin utilizar el test para seleccionarlo.

### Calidad del target
Cuantifica las inconsistencias entre `Profit` y `Sales_Amount - Cost_Amount` y mide su efecto sobre las métricas sin eliminar registros después de observar los resultados.

### Importancia por permutación
Identifica qué variables aportan mayor información predictiva.

### Comparación de conjuntos reducidos
Comprueba si gran parte de la señal se concentra en unas pocas variables.

### Análisis por dimensiones
Traduce el modelo al lenguaje de negocio: condiciones de la venta, producto, cliente, canales y periodos.

### Regla de alerta
Convierte el margen predicho en una decisión operativa configurable.

### Simulación
Permite explorar escenarios de descuento, dejando explícito que predicción no equivale a causalidad ni a elasticidad específica por producto.

---

## Qué debe reportarse al finalizar la ejecución

1. estrategia de imputación seleccionada y justificación;
2. escalado seleccionado y justificación;
3. modelo ganador y comparación con Dummy;
4. MAE, RMSE y R² en test;
5. número de registros financieramente inconsistentes;
6. métricas del modelo sobre registros consistentes vs. inconsistentes;
7. variables más importantes;
8. cuánto desempeño conserva un conjunto reducido de variables;
9. dimensión más asociada con la rentabilidad;
10. interpretación de Q4 después de conocer producto y descuento;
11. desempeño de la alerta de margen < 15 %;
12. limitaciones de la simulación de descuentos.

> **Ninguno de estos resultados, por sí solo, demuestra causalidad.**